# 1. Data Loading and Descriptive Statistics

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    from pykalman import KalmanFilter
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'pykalman'])
    from pykalman import KalmanFilter

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

In [ ]:
df_daily = pd.read_csv('dataset_daily.csv', parse_dates=['observation_date'])
df_weekly = pd.read_csv('dataset_weekly.csv', parse_dates=['observation_date'])
df_monthly = pd.read_csv('dataset_monthly.csv', parse_dates=['observation_date'])

print(f"Daily:   {df_daily.shape[0]} rows, {df_daily.shape[1]} cols, "
      f"{df_daily['observation_date'].min().date()} to {df_daily['observation_date'].max().date()}")
print(f"Weekly:  {df_weekly.shape[0]} rows, {df_weekly.shape[1]} cols")
print(f"Monthly: {df_monthly.shape[0]} rows, {df_monthly.shape[1]} cols")
df_daily.head()

In [ ]:
desc_cols = ['Brent_Price', 'WTI_Price', 'oil_return', 'OVX_Close',
             'DTWEXBGS', 'EFFR', 'Gas_Price', 'Gold_Price']
desc = df_daily[desc_cols].describe().T
desc['skew'] = df_daily[desc_cols].skew()
desc['kurtosis'] = df_daily[desc_cols].kurtosis()
desc.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_daily['observation_date'], df_daily['WTI_Price'], color='steelblue', linewidth=1)
ax.set_xlabel('Date')
ax.set_ylabel('WTI Price (USD/barrel)')
ax.set_title('WTI Crude Oil Price (2020\u20132026)')

events = {
    '2020-04-20': 'COVID Crash',
    '2022-03-08': 'Russia-Ukraine War',
    '2026-03-10': 'Recent Volatility'
}
for date_str, label in events.items():
    date = pd.Timestamp(date_str)
    if date in df_daily['observation_date'].values:
        price = df_daily.loc[df_daily['observation_date'] == date, 'WTI_Price'].values[0]
    else:
        idx = (df_daily['observation_date'] - date).abs().idxmin()
        price = df_daily.loc[idx, 'WTI_Price']
    ax.annotate(label, xy=(date, price), xytext=(0, 20),
                textcoords='offset points', fontsize=9,
                arrowprops=dict(arrowstyle='->', color='red'),
                ha='center', color='red')
plt.tight_layout()
plt.show()

# 2. Shannon Entropy \u2014 Rolling Window

In [ ]:
def rolling_shannon_entropy(series, window=60, n_bins=20):
    """
    Compute rolling Shannon Entropy on a return series.

    For each window of `window` observations, discretize returns into
    `n_bins` equal-width bins and compute Shannon entropy (base 2).

    Parameters
    ----------
    series : pd.Series \u2014 return series
    window : int \u2014 rolling window size (default 60 trading days)
    n_bins : int \u2014 number of bins for discretization (default 20)

    Returns
    -------
    pd.Series \u2014 entropy values (first window-1 entries are NaN)
    """
    entropy_values = []
    for i in range(len(series)):
        if i < window - 1:
            entropy_values.append(np.nan)
            continue
        window_data = series.iloc[i - window + 1 : i + 1].values
        counts, _ = np.histogram(window_data, bins=n_bins)
        probs = counts / counts.sum()
        probs = probs[probs > 0]
        H = -np.sum(probs * np.log2(probs))
        entropy_values.append(H)
    return pd.Series(entropy_values, index=series.index)

In [ ]:
df_daily['rolling_entropy'] = rolling_shannon_entropy(df_daily['oil_return'], window=60, n_bins=20)

print(f"Entropy range: {df_daily['rolling_entropy'].min():.4f} to {df_daily['rolling_entropy'].max():.4f}")
print(f"Entropy mean:  {df_daily['rolling_entropy'].mean():.4f}")
print(f"NaN count:     {df_daily['rolling_entropy'].isna().sum()}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

color_price = 'steelblue'
ax1.set_xlabel('Date')
ax1.set_ylabel('WTI Price (USD/barrel)', color=color_price)
ax1.plot(df_daily['observation_date'], df_daily['WTI_Price'],
         color=color_price, linewidth=1, alpha=0.8)
ax1.tick_params(axis='y', labelcolor=color_price)

ax2 = ax1.twinx()
color_entropy = 'darkorange'
ax2.set_ylabel('Rolling Shannon Entropy (60-day)', color=color_entropy)
ax2.plot(df_daily['observation_date'], df_daily['rolling_entropy'],
         color=color_entropy, linewidth=1, alpha=0.8)
ax2.tick_params(axis='y', labelcolor=color_entropy)

fig.suptitle('WTI Crude Oil Price and Rolling Shannon Entropy', fontsize=14)
fig.tight_layout()
plt.show()

# 3. Kalman Filter \u2014 State-Space Model and Regime Detection

In [ ]:
df_kf = df_daily.dropna(subset=['rolling_entropy']).copy()

obs_cols = ['rolling_entropy', 'OVX_Close', 'abs_return']
df_kf['abs_return'] = df_kf['oil_return'].abs()

for col in obs_cols:
    mean_val = df_kf[col].mean()
    std_val = df_kf[col].std()
    df_kf[f'{col}_z'] = (df_kf[col] - mean_val) / std_val

obs_cols_z = [f'{col}_z' for col in obs_cols]
observations = df_kf[obs_cols_z].values

print(f"Observation matrix shape: {observations.shape}")
print(f"Date range: {df_kf['observation_date'].min().date()} to {df_kf['observation_date'].max().date()}")

In [ ]:
kf = KalmanFilter(
    n_dim_state=1,
    n_dim_obs=3,
    em_vars=['transition_matrices',
             'observation_matrices',
             'transition_covariance',
             'observation_covariance',
             'initial_state_mean',
             'initial_state_covariance'],
)

kf = kf.em(observations, n_iter=10)

print("Estimated parameters:")
print(f"  A (transition matrix):  {kf.transition_matrices.flatten()}")
print(f"  C (observation matrix): {kf.observation_matrices.flatten()}")
print(f"  Q (state noise var):    {kf.transition_covariance.flatten()}")
print(f"  R diag (obs noise var): {np.diag(kf.observation_covariance)}")

In [ ]:
filtered_state_means, filtered_state_covs = kf.filter(observations)
hidden_state = filtered_state_means.flatten()

df_kf['hidden_state'] = hidden_state

p33 = np.percentile(hidden_state, 33)
p67 = np.percentile(hidden_state, 67)

df_kf['regime_label'] = 0
df_kf.loc[df_kf['hidden_state'] >= p33, 'regime_label'] = 1
df_kf.loc[df_kf['hidden_state'] >= p67, 'regime_label'] = 2

print(f"Regime thresholds: P33 = {p33:.4f}, P67 = {p67:.4f}")
print(f"\nRegime distribution:")
print(df_kf['regime_label'].value_counts().sort_index())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_kf['observation_date'], df_kf['hidden_state'],
        color='navy', linewidth=1, label='Hidden State')
ax.axhline(y=p33, color='orange', linestyle='--', linewidth=1, label=f'P33 = {p33:.2f}')
ax.axhline(y=p67, color='red', linestyle='--', linewidth=1, label=f'P67 = {p67:.2f}')
ax.fill_between(df_kf['observation_date'], ax.get_ylim()[0], p33, alpha=0.1, color='green')
ax.fill_between(df_kf['observation_date'], p33, p67, alpha=0.1, color='orange')
ax.fill_between(df_kf['observation_date'], p67, ax.get_ylim()[1], alpha=0.1, color='red')
ax.set_xlabel('Date')
ax.set_ylabel('Hidden State (Market Stress Index)')
ax.set_title('Kalman Filter Hidden State with Regime Thresholds')
ax.legend()
plt.tight_layout()
plt.show()

# 4. Regime Analysis and Visualization

In [ ]:
regime_colors = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
regime_labels = {0: 'Regime 0 (Calm)', 1: 'Regime 1 (Transition)', 2: 'Regime 2 (Crisis)'}

fig, ax = plt.subplots(figsize=(14, 6))

for regime in [0, 1, 2]:
    mask = df_kf['regime_label'] == regime
    dates = df_kf.loc[mask, 'observation_date']
    for d in dates:
        ax.axvspan(d - pd.Timedelta(days=0.5), d + pd.Timedelta(days=0.5),
                   alpha=0.3, color=regime_colors[regime], linewidth=0)

ax.plot(df_kf['observation_date'], df_kf['WTI_Price'],
        color='black', linewidth=1, zorder=5)

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=regime_colors[r], alpha=0.4, label=regime_labels[r])
                  for r in [0, 1, 2]]
ax.legend(handles=legend_handles, loc='upper left')
ax.set_xlabel('Date')
ax.set_ylabel('WTI Price (USD/barrel)')
ax.set_title('WTI Price with Detected Market Regimes')
plt.tight_layout()
plt.show()

In [ ]:
regime_stats = df_kf.groupby('regime_label').agg(
    count=('WTI_Price', 'size'),
    mean_return=('oil_return', 'mean'),
    std_return=('oil_return', 'std'),
    mean_OVX=('OVX_Close', 'mean'),
    std_OVX=('OVX_Close', 'std'),
    mean_entropy=('rolling_entropy', 'mean'),
    std_entropy=('rolling_entropy', 'std'),
    mean_WTI=('WTI_Price', 'mean'),
    std_WTI=('WTI_Price', 'std'),
).round(4)

def avg_consecutive_duration(series, regime):
    """Average number of consecutive days in a given regime."""
    is_regime = (series == regime).astype(int)
    blocks = (is_regime.diff().fillna(0) != 0).cumsum()
    regime_blocks = blocks[is_regime == 1]
    if len(regime_blocks) == 0:
        return 0
    durations = regime_blocks.value_counts()
    return durations.mean()

for r in [0, 1, 2]:
    regime_stats.loc[r, 'avg_duration_days'] = avg_consecutive_duration(
        df_kf['regime_label'], r)

regime_stats.index = ['Regime 0 (Calm)', 'Regime 1 (Transition)', 'Regime 2 (Crisis)']
regime_stats

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax_i, (col, title) in enumerate([
    ('oil_return', 'Oil Return'),
    ('OVX_Close', 'OVX (Implied Volatility)'),
    ('rolling_entropy', 'Shannon Entropy')
]):
    data_to_plot = [df_kf.loc[df_kf['regime_label'] == r, col].dropna() for r in [0, 1, 2]]
    bp = axes[ax_i].boxplot(data_to_plot, labels=['Calm', 'Transition', 'Crisis'],
                            patch_artist=True)
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    axes[ax_i].set_title(title)

fig.suptitle('Distribution of Key Variables by Regime', fontsize=14)
plt.tight_layout()
plt.show()

# 5. Data Merging and Export

In [ ]:
df_merged = df_kf.copy()

df_merged = pd.merge_asof(
    df_merged.sort_values('observation_date'),
    df_weekly.sort_values('observation_date'),
    on='observation_date',
    direction='backward'
)

df_merged = pd.merge_asof(
    df_merged.sort_values('observation_date'),
    df_monthly.sort_values('observation_date'),
    on='observation_date',
    direction='backward'
)

print(f"Merged dataset shape: {df_merged.shape}")
print(f"Columns: {list(df_merged.columns)}")
print(f"\nNull counts in merged data:")
print(df_merged.isnull().sum()[df_merged.isnull().sum() > 0])

In [ ]:
df_merged['WTI_Price_next'] = df_merged['WTI_Price'].shift(-1)

df_export = df_merged.dropna().copy()

export_cols = [
    'observation_date',
    'Brent_Price', 'DTWEXBGS', 'EFFR', 'Gas_Price', 'Gold_Price',
    'OVX_Close', 'WTI_Price', 'oil_return',
    'WCESTUS1', 'supply_shock', 'supply_shock_pct',
    'GPR', 'GPRT', 'GPRC', 'GEPUCURRENT', 'CPILFESL',
    'INDPRO', 'industrial_production_growth',
    'rolling_entropy', 'hidden_state', 'regime_label',
    'WTI_Price_next'
]

df_export = df_export[export_cols]
df_export.to_csv('dataset_with_regime.csv', index=False)

print(f"Exported {df_export.shape[0]} rows, {df_export.shape[1]} columns")
print(f"Date range: {df_export['observation_date'].min()} to {df_export['observation_date'].max()}")
print(f"\nRegime distribution in exported data:")
print(df_export['regime_label'].value_counts().sort_index())